# S=7 support sweep — extra seed 43 (all six training supports × both models = 12 runs)

The notebook-23 setup again, with training seed 43.
* Identity fillers, Transformer AR and causal encoder.
* 20,000 updates.
* Sampled evaluation of the LAST checkpoint.

**What the seed changes.** The training seed changes the initialization and the training stream. The held-out
grids, the validation set and the sampled test records are the same as for seed 42 (fixed evaluation seed), so the
three seeds are evaluated on identical records.

**Parallel notebooks.** 27a runs seed 43 and 27b runs seed 44; each covers all six supports
(k02 … k0234567) × {Transformer AR, causal encoder}. Run both at the same time, about 3 hours each. Seed 42 is the
existing notebook-23 runs, which are reused and not retrained.

**Report.** The last cell, rerun in either notebook once both are finished, makes the multi-seed report
`S7_Sweep_MultiSeed_Review.ipynb`:
* accuracy vs test k per training support (mean over seeds, min–max band, individual seeds);
* singleton / +1 / +2 step accuracy vs longest trained k;
* mean ± std tables.


In [ ]:
# --- 2. Bootstrap: locate or clone the PRIVATE repository (credentials are never printed or stored) ---
import base64, importlib.util, os, pathlib, subprocess, sys

REPO_URL = "https://github.com/zhexuandliu/Decompositionality.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

def _find_repo():
    candidates = [pathlib.Path("/content/Decompositionality"), pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    for p in candidates:
        if (p / "src" / "decompositionality").is_dir():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        import getpass
        token = getpass.getpass("GitHub token with read access to the private repo (input hidden): ")
    # Pass the credential through environment-scoped git config (not the URL, not the command line,
    # not persisted in .git/config).
    _hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    _env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
                GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
    REPO_DIR = pathlib.Path("/content/Decompositionality" if IN_COLAB else "Decompositionality").resolve()
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                        env=_env, capture_output=True, text=True)
    del token, _hdr, _env
    if _r.returncode != 0:
        raise RuntimeError("git clone failed (check the GITHUB_TOKEN secret / repo access):\n" + _r.stderr[-2000:])

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for _mod in ("numpy", "pandas", "matplotlib", "torch"):
    if importlib.util.find_spec(_mod) is None:  # Colab already ships these; never reinstall torch/CUDA here
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _mod], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository:", REPO_DIR, "| commit:", REPO_COMMIT or "unknown")


In [ ]:
# --- CONFIGURATION -------------------------------------------------------------------------------------------------
MODE = os.environ.get("DECOMP_MODE", "full")
SEED = 43                                   # TRAINING seed (initialization + training stream)
RUN_CONDITIONS = ['S7_k02', 'S7_k023', 'S7_k0234', 'S7_k02345', 'S7_k023456', 'S7_k0234567']
RUN_MODELS = ["transformer_ar", "causal_encoder"]
STEPS = 20000
SUBDIR = "S7_support_sweep_20k_v1"                # same folder as notebook 23 (runs live in seed_<SEED> subfolders)
EVAL_PLAN = {"n_single_grids": 1000, "n_seen_per_k": 5000, "n_long_per_k": 20000}
ALL_SEEDS = [42, 43, 44]
USE_DRIVE = IN_COLAB
RESULTS_ROOT = os.environ.get("DECOMP_RESULTS_ROOT") or (
    "/content/drive/MyDrive/decompositionality_results" if USE_DRIVE else str(REPO_DIR / "results"))
DEVICE = os.environ.get("DECOMP_DEVICE", "auto")
# -----------------------------------------------------------------------------------------------------------------
if USE_DRIVE and IN_COLAB and RESULTS_ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
import pandas as pd
from IPython.display import display
pd.set_option("display.max_rows", 500); pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)
from decompositionality import longk as LK, longk_exp as LX
print("training seed", SEED, "| evaluation seed (fixed)", LX.EVAL_SEED, "| results ->", LX.family_root(RESULTS_ROOT, MODE, SUBDIR))


In [ ]:
# --- Train (20,000 updates) + sampled LAST-checkpoint evaluation (resumable) ---------------------------------------
status = LX.run_queue(RESULTS_ROOT, MODE, SEED, RUN_CONDITIONS, RUN_MODELS, repo_dir=str(REPO_DIR), device=DEVICE,
                      subdir=SUBDIR, steps=STEPS if MODE == "full" else None, sampled_plan=EVAL_PLAN)
display(status)


In [ ]:
# --- Multi-seed report over everything finished so far (rerun once all four notebooks are done) --------------------
rp = LX.multiseed_report(RESULTS_ROOT, MODE, ALL_SEEDS, LK.S7_SWEEP, SUBDIR)
print("REVIEW FILE TO SEND BACK:", rp)
if IN_COLAB and rp:
    from google.colab import files
    files.download(str(rp))
